# YOLOv9 Colab Training Pipeline

This notebook uploads a YOLO dataset ZIP, trains YOLOv9, evaluates the held-out test split, and saves detection metrics.

Before running Cell 3, create a ZIP whose top-level folder is `dataset_yolov9` and contains `data.yaml`, `train`, `val`, and `test`.

## Dataset ZIP structure

```text
dataset_yolov9/
  data.yaml
  train/images/
  train/labels/
  val/images/
  val/labels/
  test/images/
  test/labels/
```

Upload `dataset_yolov9.zip` to the root of Google Drive (`My Drive`) before running Cell 3. The notebook mounts Drive and reads it from:

```text
/content/drive/MyDrive/dataset_yolov9.zip
```

If you store it in a Drive subfolder, update `zip_path` in Cell 3.

In [ ]:
from google.colab import drive
from pathlib import Path

# Upload dataset_yolov9.zip to the root of Google Drive before running this cell.
drive.mount('/content/drive')
zip_path = Path('/content/drive/MyDrive/dataset_yolov9.zip')

if not zip_path.exists():
    raise FileNotFoundError(
        f'ZIP not found at {zip_path}. Upload dataset_yolov9.zip to My Drive first.'
    )

zip_name = str(zip_path)
print(f'Using dataset ZIP: {zip_path}')

In [ ]:
from pathlib import Path
import shutil
import zipfile

extract_root = Path('/content/dataset_extract')
dataset_root = Path('/content/NotAISlop/dataset_yolov9')

if extract_root.exists():
    shutil.rmtree(extract_root)
if dataset_root.exists():
    shutil.rmtree(dataset_root)

extract_root.mkdir(parents=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(extract_root)

candidate = extract_root / 'dataset_yolov9'
if not (candidate / 'data.yaml').exists():
    yaml_candidates = list(extract_root.rglob('data.yaml'))
    if len(yaml_candidates) != 1:
        raise FileNotFoundError('Could not uniquely locate dataset_yolov9/data.yaml in the ZIP')
    candidate = yaml_candidates[0].parent

dataset_root.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(candidate, dataset_root)
print(f'Dataset extracted to: {dataset_root}')

In [ ]:
import yaml

with (dataset_root / 'data.yaml').open() as handle:
    data_config = yaml.safe_load(handle)

required_splits = ('train', 'val', 'test')
for split in required_splits:
    image_dir = dataset_root / split / 'images'
    label_dir = dataset_root / split / 'labels'
    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError(f'Missing {split}/images or {split}/labels')
    image_count = sum(1 for path in image_dir.iterdir() if path.is_file())
    label_count = len(list(label_dir.glob('*.txt')))
    print(f'{split}: {image_count} images, {label_count} label files')

print('Classes:', data_config.get('names'))
print((dataset_root / 'data.yaml').read_text())

In [ ]:
!pip install -q --upgrade ultralytics scipy opencv-python-headless requests

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Select Runtime > Change runtime type > T4 GPU, then rerun this notebook.')

print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Training settings

Use `EPOCHS = 30` for a quick pipeline test. Change it to `150` for the final training run. If a T4 runs out of memory, reduce `BATCH` from `4` to `2`.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

EPOCHS = 150
IMAGE_SIZE = 640
BATCH = 4
WORKERS = 2

data_yaml = dataset_root / 'data.yaml'
model = YOLO('yolov9c.pt')

train_results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    workers=WORKERS,
    device=0,
    patience=25,
    amp=True,
    cls=1.5,
    mosaic=1.0,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    degrees=0.0,
    hsv_v=0.2,
    project='/content/runs/detect',
    name='yolov9c_colab',
)

run_dir = Path(model.trainer.save_dir)
best_weights = run_dir / 'weights' / 'best.pt'
if not best_weights.exists():
    raise FileNotFoundError(best_weights)
print(f'Best model: {best_weights}')

In [ ]:
import json

CLASS_NAMES = ['shipwreck', 'pipe', 'cylinder', 'ghost_gear', 'clutter']
EVAL_CONFIDENCE = 0.001
BACKGROUND_CONFIDENCE = 0.25

best_model = YOLO(str(best_weights))
test_metrics = best_model.val(
    data=str(data_yaml),
    split='test',
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    workers=WORKERS,
    conf=EVAL_CONFIDENCE,
    plots=True,
    save_json=True,
    project=str(run_dir.parent),
    name=f'{run_dir.name}_test',
    exist_ok=True,
)

precision = float(test_metrics.box.mp)
recall = float(test_metrics.box.mr)
f1_score = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
per_class_ap = {name: None for name in CLASS_NAMES}
for class_index, ap50 in zip(test_metrics.box.ap_class_index, test_metrics.box.ap50):
    per_class_ap[CLASS_NAMES[int(class_index)]] = float(ap50)

test_image_dir = dataset_root / 'test' / 'images'
test_label_dir = dataset_root / 'test' / 'labels'
image_suffixes = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
background_images = [
    path for path in sorted(test_image_dir.iterdir())
    if path.suffix.lower() in image_suffixes
    and (not (test_label_dir / f'{path.stem}.txt').exists()
         or not (test_label_dir / f'{path.stem}.txt').read_text().strip())
]

background_false_positives = 0
for prediction in best_model.predict(
    source=background_images,
    imgsz=IMAGE_SIZE,
    conf=BACKGROUND_CONFIDENCE,
    stream=True,
    verbose=False,
):
    background_false_positives += len(prediction.boxes)

evaluation_dir = Path(test_metrics.save_dir)
report = {
    'weights': str(best_weights),
    'split': 'test',
    'mAP@0.5': float(test_metrics.box.map50),
    'mAP@0.5:0.95': float(test_metrics.box.map),
    'precision': precision,
    'recall': recall,
    'f1_score': f1_score,
    'per_class_AP@0.5': per_class_ap,
    'background_images': len(background_images),
    'background_false_positives': background_false_positives,
    'background_false_positives_per_image': (background_false_positives / len(background_images) if background_images else 0.0),
    'background_confidence_threshold': BACKGROUND_CONFIDENCE,
    'confusion_matrix': str(evaluation_dir / 'confusion_matrix.png'),
    'normalized_confusion_matrix': str(evaluation_dir / 'confusion_matrix_normalized.png'),
}

report_path = evaluation_dir / 'test_metrics.json'
report_path.write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print(f'Report saved to: {report_path}')

In [ ]:
from google.colab import files

files.download(str(report_path))
files.download(str(best_weights))